---------------------------------------------------------------------------------------------------------------------------------------
# DATA ENGINEERING
---------------------------------------------------------------------------------------------------------------------------------------

In [81]:
import pandas as pd
df = pd.read_csv("Loan_default.csv")
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [82]:
loan_simple_random = df.sample(n=1000, random_state=42)

In [83]:
%pip install -q scikit-learn

from sklearn.model_selection import train_test_split

unused, df_loan_stratified = train_test_split(
    df, test_size=0.1, stratify=df["CreditScore"], random_state=123)

df_loan_stratified

Note: you may need to restart the kernel to use updated packages.


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
92136,PXKQ8D6MGE,50,120851,246100,456,98,4,16.69,60,0.25,High School,Full-time,Married,No,No,Business,No,0
153245,2CKZX4H1TR,30,82505,166835,694,91,4,11.02,48,0.23,High School,Part-time,Married,Yes,No,Business,No,0
168967,O6X4FV5KMZ,26,132616,25164,764,109,3,24.07,60,0.50,PhD,Self-employed,Divorced,No,Yes,Auto,Yes,0
15990,LN7JI12WTY,68,62937,208054,470,96,4,13.63,36,0.54,High School,Full-time,Single,No,Yes,Education,No,0
122720,J9LVL3TBHV,57,119094,43876,582,91,2,16.18,24,0.14,Master's,Full-time,Single,Yes,No,Business,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225265,42NDPZ388Y,55,26379,220385,799,7,1,12.55,24,0.82,Bachelor's,Part-time,Married,No,No,Home,No,0
108620,ZCR4E4B65P,35,53688,40640,352,106,1,17.67,60,0.77,Bachelor's,Self-employed,Divorced,No,No,Home,No,0
55128,TLHHK7PS9A,42,75447,118494,536,4,1,6.56,48,0.90,Bachelor's,Part-time,Married,Yes,No,Other,Yes,0
76381,9MDLBDK7ZG,48,62668,25404,379,60,3,5.83,24,0.67,Bachelor's,Part-time,Single,Yes,No,Education,No,0


In [84]:
loan_simple_random["CreditScore"].value_counts(normalize=True)

CreditScore
350    0.007
403    0.006
655    0.006
320    0.006
568    0.005
       ...  
748    0.001
404    0.001
609    0.001
788    0.001
511    0.001
Name: proportion, Length: 455, dtype: float64

In [85]:
df["CreditScore"].value_counts(normalize=True)

CreditScore
630    0.002068
445    0.002040
829    0.002036
753    0.002033
670    0.002017
         ...   
629    0.001598
706    0.001590
536    0.001590
720    0.001574
724    0.001535
Name: proportion, Length: 550, dtype: float64

In [86]:
df_loan_stratified["CreditScore"].value_counts(normalize=True)


CreditScore
630    0.002076
445    0.002036
670    0.002036
829    0.002036
753    0.002036
         ...   
526    0.001606
308    0.001606
416    0.001606
720    0.001566
724    0.001527
Name: proportion, Length: 550, dtype: float64

In [87]:
tiers = [
    {"min": 750, "max": 999, "mult": 300, "cap": 1_000_000},
    {"min": 700, "max": 749, "mult": 200, "cap": 300_000},
    {"min": 650, "max": 699, "mult": 150, "cap": 150_000},
    {"min": 600, "max": 649, "mult": 100, "cap": 50_000},
    {"min": -1,  "max": 599, "mult": 50,  "cap": 20_000},
]

def get_params(score):
    for t in tiers:
        if t["min"] <= score <= t["max"]:
            return t["mult"], t["cap"]
    return 50, 20_000

def max_loan(score):
    mult, cap = get_params(score)
    return min(cap, score * mult)

df["base_MaxLoan"] = df["CreditScore"].apply(max_loan)
df["is_Exceeded"] = (df["LoanAmount"] > df["base_MaxLoan"]).astype(int)
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,base_MaxLoan,is_Exceeded
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,20000,1
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,20000,1
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,20000,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,148600,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,50000,0


In [88]:
df["is_Exceeded"].value_counts()

is_Exceeded
1    176346
0     79001
Name: count, dtype: int64

---------------------------------------------------------------------------------------------------------------------------------------
# FEATURE ENGINEEERING 
---------------------------------------------------------------------------------------------------------------------------------------

In [89]:

print(loan_simple_random.isnull().sum())

LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64


In [90]:
print(loan_simple_random.dtypes)

LoanID             object
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education          object
EmploymentType     object
MaritalStatus      object
HasMortgage        object
HasDependents      object
LoanPurpose        object
HasCoSigner        object
Default             int64
dtype: object


In [91]:
loan_simple_random.rename(columns={
    'LoanID': 'Loan_ID',
    'LoanAmount': 'Loan_Amount',
    'Income': 'Income(USD)',
    'CreditScore': 'Credit_Score',
    'InterestRate': 'Interest_Rate(%)',
    'DTIratio': 'DebtToIncomeRatio',
    'LoanTerm': 'Loan_Term(months)',
}, inplace=True)
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,base_MaxLoan,is_Exceeded
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,20000,1
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,20000,1
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,20000,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,148600,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,50000,0


In [92]:
selected_columns = [
    "MonthsEmployed",
    "Education",
    "MaritalStatus",
    "HasDependents",
    "Default",
    "LoanPurpose",
    "HasCoSigner",
    "Loan_ID"
]

data = loan_simple_random.drop(columns=selected_columns)
data

,Age,Income(USD),Loan_Amount,Credit_Score,NumCreditLines,Interest_Rate(%),Loan_Term(months),DTIRatio,EmploymentType,HasMortgage
51139,55,112656,92393,581,2,23.54,36,0.15,Self-employed,Yes
71005,56,91569,131575,641,1,15.19,12,0.43,Part-time,Yes
35684,26,78169,75417,569,3,18.02,12,0.29,Part-time,Yes
174087,26,63033,10804,326,1,14.71,24,0.41,Part-time,No
137952,24,29665,21182,662,3,15.02,60,0.69,Unemployed,No
...,...,...,...,...,...,...,...,...,...,...
55779,21,35959,200465,310,3,23.76,48,0.47,Full-time,Yes
217609,32,57400,68970,574,2,16.47,60,0.87,Unemployed,Yes
247012,38,47736,200653,705,2,12.44,48,0.47,Self-employed,Yes
117767,18,93942,202636,820,4,13.04,48,0.66,Self-employed,Yes


In [93]:
# One-hot Encoding (แก้คำเป็นตัวเลข)
#EmploymentType 
employment_dummies = pd.get_dummies(data['EmploymentType'], prefix='EmploymentType')
data = pd.concat([data.drop('EmploymentType', axis=1), employment_dummies], axis=1)
data.head()

,Age,Income(USD),Loan_Amount,Credit_Score,NumCreditLines,Interest_Rate(%),Loan_Term(months),DTIRatio,HasMortgage,EmploymentType_Full-time,EmploymentType_Part-time,EmploymentType_Self-employed,EmploymentType_Unemployed
51139,55,112656,92393,581,2,23.54,36,0.15,Yes,False,False,True,False
71005,56,91569,131575,641,1,15.19,12,0.43,Yes,False,True,False,False
35684,26,78169,75417,569,3,18.02,12,0.29,Yes,False,True,False,False
174087,26,63033,10804,326,1,14.71,24,0.41,No,False,True,False,False
137952,24,29665,21182,662,3,15.02,60,0.69,No,False,False,False,True


In [ ]:
# แปลง HasMortagage เป็น 0,1
data['HasMortgage'] = data['HasMortgage'].replace({"Yes": True, "No": False})
data.head()

C:\Users\Zajow\AppData\Local\Temp\ipykernel_27056\803876123.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['HasMortgage'] = data['HasMortgage'].replace({"Yes": True, "No": False})


,Age,Income(USD),Loan_Amount,Credit_Score,NumCreditLines,Interest_Rate(%),Loan_Term(months),DTIRatio,HasMortgage,EmploymentType_Full-time,EmploymentType_Part-time,EmploymentType_Self-employed,EmploymentType_Unemployed
51139,55,112656,92393,581,2,23.54,36,0.15,True,False,False,True,False
71005,56,91569,131575,641,1,15.19,12,0.43,True,False,True,False,False
35684,26,78169,75417,569,3,18.02,12,0.29,True,False,True,False,False
174087,26,63033,10804,326,1,14.71,24,0.41,False,False,True,False,False
137952,24,29665,21182,662,3,15.02,60,0.69,False,False,False,False,True


In [95]:
print(df.describe())


                 Age         Income     LoanAmount    CreditScore  \
count  255347.000000  255347.000000  255347.000000  255347.000000   
mean       43.498306   82499.304597  127578.865512     574.264346   
std        14.990258   38963.013729   70840.706142     158.903867   
min        18.000000   15000.000000    5000.000000     300.000000   
25%        31.000000   48825.500000   66156.000000     437.000000   
50%        43.000000   82466.000000  127556.000000     574.000000   
75%        56.000000  116219.000000  188985.000000     712.000000   
max        69.000000  149999.000000  249999.000000     849.000000   

       MonthsEmployed  NumCreditLines   InterestRate       LoanTerm  \
count   255347.000000   255347.000000  255347.000000  255347.000000   
mean        59.541976        2.501036      13.492773      36.025894   
std         34.643376        1.117018       6.636443      16.969330   
min          0.000000        1.000000       2.000000      12.000000   
25%         30.000000  